model.batch() sends multiple independent prompts to Gemini at the exact same time (in parallel) instead of sending them one by one in a for loop.Why batch() Matters

Without batch() (Sequential): If you have 3 questions and each takes 2 seconds, running model.invoke() three times in a loop takes $2 + 2 + 2 = 6$ seconds.

With batch() (Parallel): LangChain fires all 3 requests simultaneously in parallel threads and waits for all of them to finish, taking roughly $2$ seconds total—and automatically keeps the answers in the exact same order as your input list.

In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

# Send 3 independent questions in parallel
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])

for response in responses:
    print(response.text)
    print("-" * 50)

Parrots have colorful feathers for several important biological and evolutionary reasons. While they look stunning to us, every splash of red, green, yellow, or blue serves a specific purpose in their survival and social life. 

Here are the main reasons why parrots are so colorful:

### 1. Camouflage in the Rainforest
It might seem strange that bright green, red, or yellow feathers are "camouflage," but it depends on the parrot's environment. 
* **Green parrots** (like many Amazons or parakeets) blend in remarkably well with the lush green canopy of tropical rainforests. When they sit still in the leaves, predators like hawks and eagles have a hard time spotting them.
* **Colorful accents** (like red, blue, or yellow patches) often break up the bird’s silhouette, making it harder for predators to recognize them as a distinct shape (a type of camouflage known as *disruptive coloration*).

### 2. Communication and Social Status
Parrots are extremely social, intelligent flock animals. Th

With config={"max_concurrency": 2}, LangChain runs at most 2 prompts at the exact same time, even if you pass 4 (or 40) prompts in the list.

Here is how it handles your 4 prompts step by step:

It immediately sends Prompt 1 and Prompt 2 together.

As soon as one of those two finishes, it sends Prompt 3 so there are still 2 active requests running.

As soon as the next one finishes, it sends Prompt 4, and finally returns all 4 answers in the exact original order.

It acts like a speed limit so you still get faster parallel execution without sending too many requests in the same second and hitting a free-tier rate limit error.

In [2]:
# Run at most 2 requests in parallel at a time
responses = model.batch(
    [
        "Why do parrots have colorful feathers?",
        "How do airplanes fly?",
        "What is quantum computing?",
        "Why is the sky blue?"
    ],
    config={"max_concurrency": 2}
)

for i, response in enumerate(responses, start=1):
    print(f"Answer {i}:\n{response.text}\n")

Answer 1:
Parrots are among the most colorful birds in the world, and their vibrant feathers serve several crucial purposes for their survival and social life. Unlike humans who might just see pretty colors, scientists view a parrot's plumage as a multi-tool. 

Here are the primary reasons why parrots have colorful feathers:

### 1. Camouflage (Believe It or Not!)
While bright red, blue, and yellow might look like they would stand out, in the dense green canopy of a tropical rainforest, a brightly colored parrot can actually be surprisingly hard to see. 
* **The "Dappled Light" Effect:** Rainforests are full of bright sunlight filtering through leaves, creating patches of intense light and deep shadow. To a predator (like a hawk or an owl), a multicolored bird can break up its outline and blend right into the chaotic background of flowers, fruits, and sunlit leaves. 
* *Note:* Many parrots, especially females or those that nest in deep tree cavities, are greener and more subtly colored

batch and stream both together

In [3]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

prompts = [
    "Give me a 1-sentence fun fact about space.",
    "Write a 150-word paragraph explaining how photosynthesis works.",
    "Name 3 programming languages and their main use case."
]

# Runs all 3 in parallel and prints whichever finishes first
for index, response in model.batch_as_completed(prompts):
    print(f"[Prompt {index + 1} finished]:")
    print(response.text)
    print("-" * 50)

[Prompt 1 finished]:
Because there is no atmosphere in space to carry sound waves, a region of space known as the Perseus cluster contains a supermassive black hole that has been churning out the musical note B-flat—stretching 57 octaves below middle C—making it the lowest known sound in the universe.
--------------------------------------------------
[Prompt 2 finished]:
Photosynthesis is the vital biological process by which green plants, algae, and some bacteria convert light energy into chemical energy. This intricate mechanism primarily takes place within the chloroplasts of plant cells, specifically utilizing a green pigment called chlorophyll to capture sunlight. During this process, plants absorb carbon dioxide from the surrounding air through tiny pores called stomata and take up water from the soil through their roots. Powered by the absorbed solar energy, a series of chemical reactions transform these raw ingredients into glucose, a simple sugar that serves as nourishment fo